# 🎯 AML 2025 Extension: Recipe-Level Task Verification

## Pipeline Overview

**SUBSTEP 0**: Create recipe-level labels (correct/incorrect)  
**SUBSTEP 1**: Run ActionFormer to predict step boundaries → Extract embeddings  
**SUBSTEP 2**: Train Transformer classifier for recipe verification  
**SUBSTEP 3**: Match steps to task graph nodes (optional)  
**SUBSTEP 4**: Train GNN models (optional)

---

## Setup

In [ ]:
!pip install torch-geometric
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-u1ac5r76
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-u1ac5r76
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=f84506cb9693753aa7492d9677102a8e5baeb2b82d94525badf93c0a1c7bfa29
  Stored in directory: /tmp/pip-ephem-wheel-cache-r45qquoo/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [ ]:
# Core imports
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Scikit-learn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from scipy.optimize import linear_sum_assignment

# PyTorch Geometric (for GNN)
try:
    from torch_geometric.data import Data, Batch
    from torch_geometric.nn import GCNConv, GATConv
    TORCH_GEOMETRIC_AVAILABLE = True
    print("✅ PyTorch Geometric loaded")
except ImportError:
    TORCH_GEOMETRIC_AVAILABLE = False
    print("⚠️  PyTorch Geometric not available - Substep 4 will not work")

# CLIP (for text encoding)
try:
    import clip
    CLIP_AVAILABLE = True
    print("✅ CLIP loaded")
except ImportError:
    CLIP_AVAILABLE = False
    print("⚠️  CLIP not available - will use placeholder text encoder")

# Matplotlib for visualization
import matplotlib.pyplot as plt
import seaborn as sns

print(f"\n✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

✅ PyTorch Geometric loaded
✅ CLIP loaded

✅ All imports successful!
PyTorch version: 2.9.0+cu126
Device: cuda


In [ ]:
# Clone the AML 2025 project repository
import os

# Clone to /content/code (matches your working setup)
%cd /content
!rm -rf code

print("📥 Cloning AML 2025 project repository...")
!git clone --recursive -b step2 https://github.com/sinamahdavi/aml-2025-mistake-detection.git code

# Change to project directory
%cd code

print("\n📂 Project structure:")
!ls -la

print("\n✅ Repository cloned successfully!")
print("   - annotations/: Step and error annotations")
print("   - er_annotations/: Data splits")
print("   - task_graphs/: Recipe task graphs")

/content
📥 Cloning AML 2025 project repository...
Cloning into 'code'...
remote: Enumerating objects: 724, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 724 (delta 7), reused 55 (delta 3), pack-reused 622 (from 1)
Receiving objects: 100% (724/724), 4.75 MiB | 36.27 MiB/s, done.
Resolving deltas: 100% (427/427), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 13.67 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'
/content/code

📂 Project s

In [ ]:
# ============================================================================
# 📥 LOAD YOUR EGOVLP FEATURES FROM GOOGLE DRIVE
# ============================================================================
# Using the EgoVLP features you already extracted in step2_baselines.ipynb

from google.colab import drive
import os

# Mount Drive if not already mounted
if not os.path.exists('/content/drive'):
    print("📁 Mounting Google Drive...")
    drive.mount('/content/drive')
    print("✅ Drive mounted!")
else:
    print("✅ Drive already mounted")

# Path to your EgoVLP features on Drive
EGOVLP_FEATURES_PATH = "/content/drive/MyDrive/AML/data/features/egovlp"

print("\n" + "="*70)
print("🔍 CHECKING YOUR EGOVLP FEATURES")
print("="*70)

if os.path.exists(EGOVLP_FEATURES_PATH):
    egovlp_files = [f for f in os.listdir(EGOVLP_FEATURES_PATH) if f.endswith('.npz')]
    print(f"✅ Found {len(egovlp_files)} EgoVLP feature files")
    print(f"   Path: {EGOVLP_FEATURES_PATH}")

    # Show first few files
    print(f"\n📋 First 5 files:")
    for i, f in enumerate(sorted(egovlp_files)[:5], 1):
        print(f"   {i}. {f}")

    print(f"\n🎯 Your features will be used for:")
    print(f"   ✅ Substep 1: Step localization (ground-truth & ActionFormer)")
    print(f"   ✅ Substep 2: Transformer baseline")
    print(f"   ✅ ActionFormer inference (if you run it)")
    print(f"   ✅ Task graph matching (Substep 3)")

    print("\n💡 NO RE-DOWNLOADING needed - using YOUR existing features!")
else:
    print(f"❌ EgoVLP features not found at: {EGOVLP_FEATURES_PATH}")
    print("\n💡 Make sure you:")
    print("   1. Ran step2_baselines.ipynb")
    print("   2. Extracted features with OPTION A")
    print("   3. Fixed naming with OPTION D")
    print("\n   Or update EGOVLP_FEATURES_PATH above to your actual path")

print("="*70)

📁 Mounting Google Drive...
Mounted at /content/drive
✅ Drive mounted!

🔍 CHECKING YOUR EGOVLP FEATURES
✅ Found 384 EgoVLP feature files
   Path: /content/drive/MyDrive/AML/data/features/egovlp

📋 First 5 files:
   1. 10_16_360p.mp4_1s_1s.npz
   2. 10_18_360p.mp4_1s_1s.npz
   3. 10_24_360p.mp4_1s_1s.npz
   4. 10_26_360p.mp4_1s_1s.npz
   5. 10_31_360p.mp4_1s_1s.npz

🎯 Your features will be used for:
   ✅ Substep 1: Step localization (ground-truth & ActionFormer)
   ✅ Substep 2: Transformer baseline
   ✅ ActionFormer inference (if you run it)
   ✅ Task graph matching (Substep 3)

💡 NO RE-DOWNLOADING needed - using YOUR existing features!


In [ ]:
# Paths - Use YOUR EgoVLP features from Drive!
FEATURES_DIR = Path(EGOVLP_FEATURES_PATH)  # Your features from Drive
ANNOTATIONS_PATH = "annotations/annotation_json/complete_step_annotations.json"
TASK_GRAPHS_DIR = "annotations/task_graphs"
SPLIT_FILE = "er_annotations/recordings_combined_splits.json"
OUTPUT_DIR = "extension_results"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
MAX_RECORDINGS = None  # Set to None for full dataset (384 videos)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_STRATEGY = "any_error"  # "any_error", "majority", or "critical"

# Model hyperparameters (OPTIMIZED FOR 384 VIDEOS)
HIDDEN_DIM = 512        # Larger capacity - 384 videos can support this
NUM_HEADS = 8           # Scaled with hidden_dim (512 / 8 = 64 per head)
NUM_LAYERS = 3          # Deeper for more complex patterns
DROPOUT = 0.2           # Moderate regularization
LEARNING_RATE = 3e-4    # Balanced learning rate
WEIGHT_DECAY = 1e-3     # Strong regularization to prevent overfitting
NUM_EPOCHS_SUBSTEP2 = 20  # Sufficient for convergence
NUM_EPOCHS_SUBSTEP4 = 20
BATCH_SIZE = 16         # Good balance for 384 videos

print(f"✅ Configuration set")
print(f"   Device: {DEVICE}")
print(f"   Features from: {FEATURES_DIR}")
print(f"   Max recordings: {MAX_RECORDINGS if MAX_RECORDINGS else 'ALL'}")
print(f"   Output directory: {OUTPUT_DIR}")

✅ Configuration set
   Device: cuda
   Features from: /content/drive/MyDrive/AML/data/features/egovlp
   Max recordings: ALL
   Output directory: extension_results


In [ ]:
# Initialize result variables (so they exist in namespace)
substep2_avg = None
substep2_results = []
all_recording_ids = []
recipe_labels = {}

print("✅ Result variables initialized")

✅ Result variables initialized


---
## 📋 SUBSTEP 0: Create Recipe-Level Labels

In [ ]:
def create_recipe_level_labels(step_annotations_path, strategy='any_error'):
    """
    Create recipe-level binary labels from step-level annotations.

    Args:
        step_annotations_path: Path to step annotations JSON
        strategy: 'any_error', 'majority', or 'critical'

    Returns:
        recipe_labels: Dict[recording_id -> {label, num_steps, num_errors}]
    """
    print(f"📂 Loading step annotations from {step_annotations_path}")

    with open(step_annotations_path, 'r') as f:
        step_annotations = json.load(f)

    recipe_labels = {}

    for recording_id, recording_data in step_annotations.items():
        steps = recording_data.get('steps', [])

        if len(steps) == 0:
            recipe_labels[recording_id] = {'label': 1, 'num_steps': 0, 'num_errors': 0}
            continue

        # Count errors
        error_count = sum(1 for step in steps if step.get('has_errors', False))
        total_steps = len(steps)

        # Determine label based on strategy
        if strategy == 'any_error':
            label = 0 if error_count > 0 else 1
        elif strategy == 'majority':
            label = 0 if error_count > (total_steps / 2) else 1
        elif strategy == 'critical':
            label = 0 if error_count >= 2 else 1
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        recipe_labels[recording_id] = {
            'label': label,
            'num_steps': total_steps,
            'num_errors': error_count,
            'error_rate': error_count / total_steps if total_steps > 0 else 0.0
        }

    # Statistics
    total_recipes = len(recipe_labels)
    correct_recipes = sum(1 for v in recipe_labels.values() if v['label'] == 1)
    incorrect_recipes = total_recipes - correct_recipes

    print(f"\n📊 Recipe-Level Label Statistics ({strategy}):")
    print(f"  Total recipes: {total_recipes}")
    print(f"  Correct (label=1): {correct_recipes} ({100*correct_recipes/total_recipes:.1f}%)")
    print(f"  Incorrect (label=0): {incorrect_recipes} ({100*incorrect_recipes/total_recipes:.1f}%)")

    return recipe_labels

# Create labels
recipe_labels = create_recipe_level_labels(ANNOTATIONS_PATH, strategy=LABEL_STRATEGY)

# Save labels
labels_path = os.path.join(OUTPUT_DIR, 'recipe_level_labels.json')
with open(labels_path, 'w') as f:
    json.dump(recipe_labels, f, indent=2)
print(f"\n💾 Saved to {labels_path}")

📂 Loading step annotations from annotations/annotation_json/complete_step_annotations.json

📊 Recipe-Level Label Statistics (any_error):
  Total recipes: 384
  Correct (label=1): 164 (42.7%)
  Incorrect (label=0): 220 (57.3%)

💾 Saved to extension_results/recipe_level_labels.json


---
## 🚀 SUBSTEP 1: ActionFormer - Step Localization

Run inference with pre-trained ActionFormer (Ego4D+EgoVLP) to predict step boundaries.

### 1.1 Clone ActionFormer Repository

In [ ]:
import os
import subprocess

# Clone ActionFormer if not already present
ACTIONFORMER_DIR = './actionformer_release'

if not os.path.exists(ACTIONFORMER_DIR):
    print("📥 Cloning ActionFormer repository...")
    subprocess.run([
        'git', 'clone',
        'https://github.com/happyharrycn/actionformer_release.git',
        ACTIONFORMER_DIR
    ], check=True)
    print("✅ Cloned successfully")
else:
    print(f"✅ ActionFormer already exists at {ACTIONFORMER_DIR}")

# Install ActionFormer dependencies
print("\n📦 Installing ActionFormer dependencies...")
subprocess.run([
    'pip', 'install', '-q',
    'pyyaml', 'yacs', 'h5py'
], check=True)
print("✅ Dependencies installed")

✅ ActionFormer already exists at ./actionformer_release

📦 Installing ActionFormer dependencies...
✅ Dependencies installed


### 1.2 Download Pre-trained Checkpoint

In [ ]:
import zipfile
from pathlib import Path

# Paths
ZIP_PATH = Path("/content/drive/MyDrive/AML/data/actionformer/ego4d_egovlp_reproduce.zip")
PRETRAINED_DIR = Path('./pretrained/ego4d_egovlp_reproduce')
CHECKPOINT_PATH = PRETRAINED_DIR / "epoch_010.pth.tar"

print("="*70)
print("📥 ActionFormer Pre-trained Checkpoint (Ego4D + EgoVLP)")
print("="*70)

# Check if checkpoint already extracted
if CHECKPOINT_PATH.exists():
    print(f"\n✅ Checkpoint already exists at: {CHECKPOINT_PATH}")

    # Verify file size (should be around 433MB)
    import os
    size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)
    print(f"   File size: {size_mb:.1f} MB")

    if size_mb < 400:
        print("   ⚠️  Warning: File size seems too small, may be corrupted")
    else:
        print("   ✅ File size looks correct")
    print("="*70)

elif ZIP_PATH.exists():
    print(f"\n📦 Found zip file at: {ZIP_PATH}")

    # Get zip file size
    import os
    zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    print(f"   Zip size: {zip_size_mb:.1f} MB")

    # Extract zip file
    print("\n🔄 Extracting checkpoint...")
    PRETRAINED_DIR.parent.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('./pretrained/')

    # Verify extraction
    if CHECKPOINT_PATH.exists():
        print(f"✅ Extraction successful!")
        print(f"   Checkpoint at: {CHECKPOINT_PATH}")

        # Verify checkpoint size
        size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)
        print(f"   Checkpoint size: {size_mb:.1f} MB")
    else:
        print(f"❌ Extraction failed - checkpoint not found at: {CHECKPOINT_PATH}")
    print("="*70)

else:
    print(f"\n❌ Zip file not found at: {ZIP_PATH}")
    print("\n📋 Please ensure you have:")
    print("   1. Downloaded ego4d_egovlp_reproduce.zip from:")
    print("      https://drive.google.com/drive/folders/1NpAECS0ZhcCuehXkF9OhLQDPFrNdStJb")
    print("   2. Uploaded it to your Google Drive at:")
    print(f"      {ZIP_PATH}")
    print("="*70)

📥 ActionFormer Pre-trained Checkpoint (Ego4D + EgoVLP)

✅ Checkpoint already exists at: pretrained/ego4d_egovlp_reproduce/epoch_010.pth.tar
   File size: 285.3 MB
   ⚠️  Warning: File size seems too small, may be corrupted


### 1.3 Prepare Data (Convert .npz → .npy)

In [ ]:
import sys
sys.path.insert(0, './actionformer_release')

# Convert YOUR EgoVLP .npz features to .npy format for ActionFormer
ACTIONFORMER_FEATURES_DIR = Path('./data/actionformer_features')
ACTIONFORMER_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("🔄 Converting YOUR EgoVLP features from .npz to .npy format...")
print(f"   Source: {FEATURES_DIR}")
print(f"   Target: {ACTIONFORMER_FEATURES_DIR}")

converted_count = 0

# Get recording IDs from split file
with open(SPLIT_FILE, 'r') as f:
    splits = json.load(f)
recording_ids = splits['train'] + splits['val'] + splits['test']

for recording_id in tqdm(recording_ids[:MAX_RECORDINGS], desc="Converting features"):
    npz_path = FEATURES_DIR / f"{recording_id}_360p.mp4_1s_1s.npz"
    npy_path = ACTIONFORMER_FEATURES_DIR / f"{recording_id}.npy"

    if npy_path.exists():
        continue

    if npz_path.exists():
        # Load YOUR .npz and save as .npy
        data = np.load(npz_path)
        features = data['arr_0']  # Shape: (T, 768)
        np.save(npy_path, features)
        converted_count += 1

print(f"✅ Converted {converted_count} feature files from YOUR Drive")
print(f"   Total available: {len(list(ACTIONFORMER_FEATURES_DIR.glob('*.npy')))} .npy files")

🔄 Converting YOUR EgoVLP features from .npz to .npy format...
   Source: /content/drive/MyDrive/AML/data/features/egovlp
   Target: data/actionformer_features


Converting features: 100%|██████████| 383/383 [00:39<00:00,  9.82it/s]

✅ Converted 363 feature files from YOUR Drive
   Total available: 383 .npy files


In [ ]:
# Load annotations and prepare data structures for ActionFormer
print("📂 Loading annotations for ActionFormer conversion...")

# Load complete step annotations
with open(ANNOTATIONS_PATH, 'r') as f:
    complete_annotations = json.load(f)

# Load step descriptions
step_desc_path = "annotations/annotation_json/step_idx_description.json"
with open(step_desc_path, 'r') as f:
    step_id_to_description = json.load(f)

# Create recording to recipe mapping
recording_to_recipe = {}
for recording_id in recording_ids[:MAX_RECORDINGS]:
    if recording_id in complete_annotations:
        # Get activity_id from annotations
        activity_id = complete_annotations[recording_id].get('activity_id', 'unknown')
        recording_to_recipe[recording_id] = str(activity_id)

print(f"✅ Loaded annotations for {len(complete_annotations)} recordings")
print(f"✅ Loaded {len(step_id_to_description)} step descriptions")
print(f"✅ Mapped {len(recording_to_recipe)} recordings to recipes")

📂 Loading annotations for ActionFormer conversion...
✅ Loaded annotations for 384 recordings
✅ Loaded 350 step descriptions
✅ Mapped 383 recordings to recipes


In [ ]:
# Convert CaptainCook4D annotations to ActivityNet format for ActionFormer
# ActivityNet format: list of dicts with 'video', 'duration', 'segments', 'labels'

print("🔄 Converting annotations to ActivityNet format...")

actionformer_annotations = {
    'database': {},
    'version': 'CaptainCook4D-v1.0',
    'taxonomy': {}
}

# Build step taxonomy
step_taxonomy = {}
for step_id, step_desc in step_id_to_description.items():
    step_taxonomy[str(step_id)] = {
        'nodeName': step_desc,
        'parentName': 'Root'
    }
actionformer_annotations['taxonomy'] = step_taxonomy

# Convert each recording
for recording_id in recording_ids[:MAX_RECORDINGS]:
    if recording_id not in complete_annotations:
        continue

    steps_data = complete_annotations[recording_id]['steps']

    # Get video duration (from features)
    npy_path = ACTIONFORMER_FEATURES_DIR / f"{recording_id}.npy"
    if not npy_path.exists():
        continue

    features = np.load(npy_path)
    duration = len(features) * 1.0  # 1 feature per second

    # Extract segments and labels
    segments = []
    labels = []

    for step in steps_data:
        if step['start_time'] >= 0 and step['end_time'] >= 0:
            segments.append([step['start_time'], step['end_time']])
            labels.append(str(step['step_id']))

    if len(segments) > 0:
        actionformer_annotations['database'][recording_id] = {
            'duration': float(duration),
            'subset': 'validation',  # We'll use for inference only
            'annotations': [
                {
                    'segment': seg,
                    'label': label
                } for seg, label in zip(segments, labels)
            ]
        }

# Save annotations
annotations_path = Path('./data/actionformer_annotations.json')
annotations_path.parent.mkdir(parents=True, exist_ok=True)

with open(annotations_path, 'w') as f:
    json.dump(actionformer_annotations, f, indent=2)

print(f"✅ Converted {len(actionformer_annotations['database'])} recordings")
print(f"   Saved to: {annotations_path}")

🔄 Converting annotations to ActivityNet format...
✅ Converted 383 recordings
   Saved to: data/actionformer_annotations.json


In [ ]:
# Create ActionFormer config file
config_yaml = """
# Dataset configuration
dataset_name: captaincook4d
train_split: ['training']
val_split: ['validation']
dataset: {
  json_file: ./data/actionformer_annotations.json,
  feat_folder: ./data/actionformer_features/,
  file_prefix: ~,
  file_ext: .npy,
  num_classes: 70,
  input_dim: 768,
  feat_stride: 1,
  num_frames: 1,
  default_fps: 1,
  trunc_thresh: 0.5,
  crop_ratio: [0.9, 1.0],
  max_seq_len: 2304,
  force_upsampling: False
}

# Model configuration (EgoVLP features)
model: {
  fpn_type: identity,
  max_buffer_len_factor: 6.0,
  n_head: 4,
  embd_kernel_size: 3,
  embd_dim: 512,
  embd_with_ln: True,
  fpn_dim: 512,
  fpn_with_ln: True,
  head_dim: 512,
  regression_range: [[0, 4], [4, 8], [8, 16], [16, 32], [32, 64], [64, 10000]],
  head_num_layers: 3,
  head_kernel_size: 3,
  head_with_ln: True,
  use_abs_pe: False,
  use_rel_pe: False
}

# Inference configuration
test_cfg: {
  pre_nms_thresh: 0.001,
  pre_nms_topk: 2000,
  iou_threshold: 0.1,
  min_score: 0.001,
  max_seg_num: 200,
  nms_method: soft,
  nms_sigma: 0.5,
  duration_thresh: 0.05,
  multiclass_nms: True,
  ext_score_file: ~,
  voting_thresh: 0.7
}

# Output
output_folder: ./ckpt/
init_rand_seed: 1234567891
"""

config_path = Path('./configs/captaincook4d_egovlp.yaml')
config_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_path, 'w') as f:
    f.write(config_yaml)

print(f"✅ Config saved to: {config_path}")

✅ Config saved to: configs/captaincook4d_egovlp.yaml


### 1.4 Run ActionFormer Inference

In [ ]:
# Build ActionFormer C++ extensions (required for NMS operations)
import os
import subprocess
from pathlib import Path

print("🔨 Building ActionFormer C++ extensions...")
print("   This compiles the NMS (Non-Maximum Suppression) module")

# Get absolute path to ActionFormer directory
actionformer_abs_path = Path(ACTIONFORMER_DIR).resolve()
utils_dir = actionformer_abs_path / 'libs' / 'utils'
setup_py_path = utils_dir / 'setup.py'

print(f"📂 ActionFormer directory: {actionformer_abs_path}")
print(f"📂 Utils directory: {utils_dir}")

# Check if setup.py exists
if not setup_py_path.exists():
    print(f"❌ setup.py not found at: {setup_py_path}")
    print(f"   Please make sure ActionFormer was cloned correctly")
else:
    print(f"✅ Found setup.py at: {setup_py_path}")

    # Save current directory
    original_dir = os.getcwd()

    try:
        # Change to libs/utils directory (as per INSTALL.md)
        os.chdir(str(utils_dir))
        print(f"📂 Changed to: {os.getcwd()}")

        # Build extensions with install --user (as per INSTALL.md)
        print("⚙️  Running: python setup.py install --user")
        result = subprocess.run(
            ['python', 'setup.py', 'install', '--user'],
            capture_output=True,
            text=True,
            check=True
        )

        print("✅ C++ extensions built and installed successfully")
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)

        # Add to Python path
        import sys
        libs_path = str(actionformer_abs_path / 'libs')
        if libs_path not in sys.path:
            sys.path.insert(0, libs_path)

        print(f"✅ Added to Python path: {libs_path}")

    except subprocess.CalledProcessError as e:
        print(f"❌ Build failed with error:")
        print(e.stderr)
        print("\n⚠️  This is required for ActionFormer inference")
        print("\n📋 Stdout:")
        print(e.stdout)

    finally:
        # Return to original directory
        os.chdir(original_dir)
        print(f"📂 Returned to: {os.getcwd()}")

print(f"\n✅ Ready for ActionFormer inference")

🔨 Building ActionFormer C++ extensions...
   This compiles the NMS (Non-Maximum Suppression) module
📂 ActionFormer directory: /content/code/actionformer_release
📂 Utils directory: /content/code/actionformer_release/libs/utils
✅ Found setup.py at: /content/code/actionformer_release/libs/utils/setup.py
📂 Changed to: /content/code/actionformer_release/libs/utils
⚙️  Running: python setup.py install --user
✅ C++ extensions built and installed successfully
hing under it)
Processing nms_1d_cpu-0.0.0-py3.12-linux-x86_64.egg
creating /root/.local/lib/python3.12/site-packages/nms_1d_cpu-0.0.0-py3.12-linux-x86_64.egg
Extracting nms_1d_cpu-0.0.0-py3.12-linux-x86_64.egg to /root/.local/lib/python3.12/site-packages
Adding nms-1d-cpu 0.0.0 to easy-install.pth file

Installed /root/.local/lib/python3.12/site-packages/nms_1d_cpu-0.0.0-py3.12-linux-x86_64.egg
Processing dependencies for nms-1d-cpu==0.0.0
Finished processing dependencies for nms-1d-cpu==0.0.0

✅ Added to Python path: /content/code/actio

In [ ]:
# Ensure nms_1d_cpu module is importable
import sys
import site

# Add user site-packages to Python path (where nms_1d_cpu was installed)
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
    print(f"✅ Added user site-packages to path: {user_site}")

# Also add ActionFormer libs to path
actionformer_libs = str(Path(ACTIONFORMER_DIR).resolve() / 'libs')
if actionformer_libs not in sys.path:
    sys.path.insert(0, actionformer_libs)
    print(f"✅ Added ActionFormer libs to path: {actionformer_libs}")

# Verify nms_1d_cpu is now importable
try:
    import nms_1d_cpu
    print("✅ nms_1d_cpu module successfully imported")
except ImportError as e:
    print(f"❌ Still cannot import nms_1d_cpu: {e}")
    print(f"\n📋 Current sys.path:")
    for p in sys.path[:10]:
        print(f"   - {p}")
    print("\n💡 You may need to restart the runtime (Runtime → Restart runtime)")
    print("   Then re-run all cells from the beginning")

✅ nms_1d_cpu module successfully imported


In [ ]:
# Diagnose and fix nms_1d_cpu import issue
import os
import sys
from pathlib import Path

print("🔍 Diagnosing nms_1d_cpu installation...")

# Check what was installed
user_site = '/root/.local/lib/python3.12/site-packages'
print(f"\n📂 Contents of {user_site}:")
if os.path.exists(user_site):
    items = os.listdir(user_site)
    for item in sorted(items):
        if 'nms' in item.lower():
            print(f"   ✓ {item}")
            full_path = os.path.join(user_site, item)
            if os.path.isdir(full_path):
                print(f"      Contents: {os.listdir(full_path)[:5]}")

# The .egg file needs to be added to sys.path
egg_file = None
for item in os.listdir(user_site):
    if item.startswith('nms_1d_cpu') and item.endswith('.egg'):
        egg_file = os.path.join(user_site, item)
        break

if egg_file and os.path.exists(egg_file):
    print(f"\n📦 Found egg file: {egg_file}")
    if egg_file not in sys.path:
        sys.path.insert(0, egg_file)
        print(f"✅ Added egg file to sys.path")

    # Try importing now
    try:
        import nms_1d_cpu
        print("✅ Successfully imported nms_1d_cpu!")
        print(f"   Module location: {nms_1d_cpu.__file__}")
    except ImportError as e:
        print(f"❌ Still cannot import: {e}")

        # Last resort: try to extract the egg
        print("\n🔧 Attempting to extract egg file...")
        import zipfile
        extract_dir = os.path.join(user_site, 'nms_1d_cpu_extracted')
        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(egg_file, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        if extract_dir not in sys.path:
            sys.path.insert(0, extract_dir)

        try:
            import nms_1d_cpu
            print("✅ Successfully imported after extracting egg!")
        except ImportError as e:
            print(f"❌ Still failed: {e}")
else:
    print("❌ No egg file found")
    print("\n⚠️  You may need to:")
    print("   1. Restart the runtime (Runtime → Restart runtime)")
    print("   2. Re-run cells from the beginning")

🔍 Diagnosing nms_1d_cpu installation...

📂 Contents of /root/.local/lib/python3.12/site-packages:
   ✓ nms_1d_cpu-0.0.0-py3.12-linux-x86_64.egg
      Contents: ['nms_1d_cpu.py', 'nms_1d_cpu.cpython-312-x86_64-linux-gnu.so', '__pycache__', 'EGG-INFO']

📦 Found egg file: /root/.local/lib/python3.12/site-packages/nms_1d_cpu-0.0.0-py3.12-linux-x86_64.egg
✅ Successfully imported nms_1d_cpu!
   Module location: /root/.local/lib/python3.12/site-packages/nms_1d_cpu-0.0.0-py3.12-linux-x86_64.egg/nms_1d_cpu.cpython-312-x86_64-linux-gnu.so


In [ ]:
# Load ActionFormer model and run inference
import torch
import torch.nn as nn

# Add ActionFormer to path
import sys
if './actionformer_release' not in sys.path:
    sys.path.insert(0, './actionformer_release')

from libs.core import load_config
from libs.modeling import make_meta_arch

print("🔧 Loading ActionFormer model...")

# Load config
config = load_config('./configs/captaincook4d_egovlp.yaml')

# Build model
model = make_meta_arch(config['model_name'], **config['model'])

# Load checkpoint
checkpoint_path = PRETRAINED_DIR / "epoch_010.pth.tar"
if checkpoint_path.exists():
    print(f"📥 Loading checkpoint: {checkpoint_path}")

    # Try with weights_only=False for compatibility with older checkpoints
    try:
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    except Exception as e:
        print(f"⚠️  Failed with weights_only=False, trying legacy mode...")
        # Fallback for even older PyTorch versions
        checkpoint = torch.load(checkpoint_path, map_location='cpu')

    model.load_state_dict(checkpoint['state_dict'], strict=False)
    print("✅ Model loaded successfully")
else:
    print(f"⚠️  Checkpoint not found at {checkpoint_path}")
    print("   Please download manually from Google Drive")
    print("   Link: https://drive.google.com/drive/folders/1uEqJp07X9l5TpZSdBpMVm0LDNTpbR17o")

# Set to eval mode
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print(f"✅ Model ready on {device}")

🔧 Loading ActionFormer model...
📥 Loading checkpoint: pretrained/ego4d_egovlp_reproduce/epoch_010.pth.tar
✅ Model loaded successfully
✅ Model ready on cuda


In [ ]:
# Run inference on all recordings
from libs.utils import batched_nms

print("🔮 Running ActionFormer inference...")

actionformer_predictions = {}

with torch.no_grad():
    for recording_id in tqdm(recording_ids[:MAX_RECORDINGS], desc="ActionFormer inference"):
        # Load features
        npy_path = ACTIONFORMER_FEATURES_DIR / f"{recording_id}.npy"
        if not npy_path.exists():
            continue

        features = np.load(npy_path)  # Shape: (T, 768)
        features_tensor = torch.from_numpy(features).float().to(device)  # (T, 768)

        # Prepare input dict (ActionFormer will add batch dimension)
        video_dict = {
            'video_id': recording_id,
            'feats': features_tensor.permute(1, 0),  # (768, T)
            'feat_stride': 1,
            'feat_num_frames': 1,
            'fps': 1.0,  # EgoVLP features are extracted at 1 fps
            'duration': features.shape[0]  # Duration in seconds
        }

        # Forward pass (model expects a list of video dicts)
        try:
            results = model([video_dict])

            # Debug: check results format
            if not isinstance(results, list) or len(results) == 0:
                print(f"⚠️  Unexpected results format for {recording_id}: {type(results)}")
                continue

            result = results[0]

            # Check if result is a dict with expected keys
            if not isinstance(result, dict):
                print(f"⚠️  Result is not a dict for {recording_id}: {type(result)}")
                continue

            if 'segments' not in result or 'scores' not in result or 'labels' not in result:
                print(f"⚠️  Missing keys in result for {recording_id}: {result.keys()}")
                continue

            segments = result['segments']  # (N, 2) [start, end]
            scores = result['scores']      # (N,)
            labels = result['labels']      # (N,)

            # Debug: show score distribution for first video
            if recording_id == recording_ids[0]:
                print(f"\n📊 Debug for {recording_id}:")
                print(f"   Total predictions: {len(scores)}")
                if len(scores) > 0:
                    print(f"   Score range: [{scores.min():.3f}, {scores.max():.3f}]")
                    print(f"   Score stats: mean={scores.mean():.3f}, median={np.median(scores):.3f}")

            # Filter by confidence threshold (Ego4D model has low confidence on CaptainCook4D)
            conf_thresh = 0.01  # Very low threshold due to domain shift
            valid_mask = scores >= conf_thresh

            if recording_id == recording_ids[0]:
                print(f"   Predictions after threshold {conf_thresh}: {valid_mask.sum()}")

            predicted_boundaries = []
            for seg, score, label in zip(segments[valid_mask],
                                         scores[valid_mask],
                                         labels[valid_mask]):
                predicted_boundaries.append({
                    'start': float(seg[0]),
                    'end': float(seg[1]),
                    'score': float(score),
                    'label': int(label)
                })

            # Sort by start time
            predicted_boundaries = sorted(predicted_boundaries, key=lambda x: x['start'])
            actionformer_predictions[recording_id] = predicted_boundaries

        except Exception as e:
            print(f"⚠️  Error processing {recording_id}: {e}")
            import traceback
            traceback.print_exc()
            continue

print(f"✅ Processed {len(actionformer_predictions)} recordings")
if len(actionformer_predictions) > 0:
    print(f"   Average predictions per video: {np.mean([len(p) for p in actionformer_predictions.values()]):.1f}")
else:
    print("   ⚠️  No predictions generated!")

🔮 Running ActionFormer inference...


ActionFormer inference:   0%|          | 1/383 [00:00<00:48,  7.87it/s]


📊 Debug for 1_19:
   Total predictions: 200
   Score range: [0.027, 0.044]
   Score stats: mean=0.030, median=0.030
   Predictions after threshold 0.01: 200


ActionFormer inference: 100%|██████████| 383/383 [00:33<00:00, 11.55it/s]

✅ Processed 383 recordings
   Average predictions per video: 200.0


### 1.5 Extract Step Embeddings

In [ ]:
# Extract step embeddings using ActionFormer predictions
print("🔧 Extracting step embeddings from ActionFormer predictions...")
print(f"   Total predictions available: {len(actionformer_predictions)}")

if len(actionformer_predictions) == 0:
    print("⚠️  No predictions found! Check if inference cell completed successfully.")

actionformer_embeddings = {}
actionformer_recording_info = {}

for recording_id in tqdm(recording_ids[:MAX_RECORDINGS], desc="Extracting embeddings"):
    if recording_id not in actionformer_predictions:
        continue

    # Load video features
    npz_path = FEATURES_DIR / f"{recording_id}_360p.mp4_1s_1s.npz"
    if not npz_path.exists():
        continue

    video_features = np.load(npz_path)['arr_0']  # (T, 768)
    predictions = actionformer_predictions[recording_id]

    if len(predictions) == 0:
        continue

    # Extract embedding for each predicted step
    step_embeddings = []
    for pred in predictions:
        start_sec = int(np.floor(pred['start']))
        end_sec = int(np.ceil(pred['end']))

        # Clip to valid range
        start_sec = max(0, start_sec)
        end_sec = min(len(video_features), end_sec)

        if start_sec >= end_sec:
            continue

        # Average features within step boundary
        step_features = video_features[start_sec:end_sec]
        step_embedding = np.mean(step_features, axis=0)  # (768,)
        step_embeddings.append(step_embedding)

    if len(step_embeddings) > 0:
        actionformer_embeddings[recording_id] = np.array(step_embeddings)  # (N_steps, 768)
        actionformer_recording_info[recording_id] = {
            'recipe_id': recording_to_recipe[recording_id],
            'label': recipe_labels[recording_id]['label'], # Access the 'label' key
            'num_steps': len(step_embeddings),
        }

# This block should be outside the loop over recording_ids
if len(actionformer_embeddings) > 0:
    print(f"   ✅ Extracted embeddings for {len(actionformer_embeddings)} recordings")
    print(f"   Average steps per recording: {np.mean([len(e) for e in actionformer_embeddings.values()]):.1f}")

    # Save embeddings
    embeddings_path = Path(OUTPUT_DIR) / 'step1_actionformer_embeddings.npz'
    np.savez(embeddings_path, **actionformer_embeddings)
    print(f"💾 Saved embeddings to {embeddings_path}")

    # Save metadata
    metadata_path = Path(OUTPUT_DIR) / 'step1_actionformer_metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(actionformer_recording_info, f, indent=2)

    print(f"💾 Saved metadata to {metadata_path}")
else:
    print("⚠️  No embeddings to save. Check inference results.")

🔧 Extracting step embeddings from ActionFormer predictions...
   Total predictions available: 383


Extracting embeddings: 100%|██████████| 383/383 [00:10<00:00, 38.22it/s]


   ✅ Extracted embeddings for 383 recordings
   Average steps per recording: 200.0
💾 Saved embeddings to extension_results/step1_actionformer_embeddings.npz
💾 Saved metadata to extension_results/step1_actionformer_metadata.json


In [ ]:
# Extract step embeddings using GROUND-TRUTH boundaries
print("🔧 Extracting step embeddings from ground-truth annotations...")

# Load ground-truth step annotations
step_annotations_path = Path('./annotations/annotation_json/step_annotations.json')
with open(step_annotations_path, 'r') as f:
    step_annotations = json.load(f)

groundtruth_embeddings = {}
groundtruth_recording_info = {}

for recording_id in tqdm(recording_ids[:MAX_RECORDINGS], desc="Extracting GT embeddings"):
    # Check if recording has annotations
    if recording_id not in step_annotations:
        continue

    # Load video features
    npz_path = FEATURES_DIR / f"{recording_id}_360p.mp4_1s_1s.npz"
    if not npz_path.exists():
        continue

    video_features = np.load(npz_path)['arr_0']  # (T, 768)
    recording_steps = step_annotations[recording_id]

    # Handle different possible JSON structures
    if isinstance(recording_steps, str):
        # If it's a string, skip
        continue
    if isinstance(recording_steps, dict):
        # If it's a dict, it might have 'steps' key or be a single step
        if 'steps' in recording_steps:
            recording_steps = recording_steps['steps']
        else:
            # Treat the dict itself as a single step
            recording_steps = [recording_steps]

    if not recording_steps or len(recording_steps) == 0:
        continue

    # Extract embedding for each ground-truth step
    step_embeddings = []
    for step in recording_steps:
        if not isinstance(step, dict):
            continue

        # Handle different key names for start/end times
        start_sec = None
        end_sec = None

        if 'start' in step and 'end' in step:
            start_sec = int(np.floor(step['start']))
            end_sec = int(np.ceil(step['end']))
        elif 'start_time' in step and 'end_time' in step:
            start_sec = int(np.floor(step['start_time']))
            end_sec = int(np.ceil(step['end_time']))
        elif 'begin' in step and 'end' in step:
            start_sec = int(np.floor(step['begin']))
            end_sec = int(np.ceil(step['end']))
        else:
            # Debug: print first step structure if keys not found
            if len(step_embeddings) == 0:
                print(f"⚠️  Unknown step format for {recording_id}: {list(step.keys())}")
            continue

        # Clip to valid range
        start_sec = max(0, start_sec)
        end_sec = min(len(video_features), end_sec)

        if start_sec >= end_sec:
            continue

        # Average features within step boundary
        step_features = video_features[start_sec:end_sec]
        step_embedding = np.mean(step_features, axis=0)  # (768,)
        step_embeddings.append(step_embedding)

    if len(step_embeddings) > 0:
        groundtruth_embeddings[recording_id] = np.array(step_embeddings)  # (N_steps, 768)
        groundtruth_recording_info[recording_id] = {
            'recipe_id': recording_to_recipe[recording_id],
            'label': recipe_labels[recording_id],
            'num_steps': len(step_embeddings),
            'video_length': len(video_features)
        }

print(f"✅ Extracted embeddings for {len(groundtruth_embeddings)} recordings")
if len(groundtruth_embeddings) > 0:
    print(f"   Average steps per recording: {np.mean([len(e) for e in groundtruth_embeddings.values()]):.1f}")

    # Save embeddings
    embeddings_path = Path(OUTPUT_DIR) / 'step1_groundtruth_embeddings.npz'
    np.savez(embeddings_path, **groundtruth_embeddings)
    print(f"💾 Saved embeddings to {embeddings_path}")

    # Save metadata
    metadata_path = Path(OUTPUT_DIR) / 'step1_groundtruth_metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(groundtruth_recording_info, f, indent=2)
    print(f"💾 Saved metadata to {metadata_path}")
else:
    print("⚠️  No embeddings to save. Check annotations.")

🔧 Extracting step embeddings from ground-truth annotations...


Extracting GT embeddings: 100%|██████████| 383/383 [00:08<00:00, 45.40it/s]

✅ Extracted embeddings for 383 recordings
   Average steps per recording: 14.1
💾 Saved embeddings to extension_results/step1_groundtruth_embeddings.npz
💾 Saved metadata to extension_results/step1_groundtruth_metadata.json


---
## 🤖 SUBSTEP 2: Transformer Classifier

Train a Transformer to classify recipe-level correctness from step sequences.

In [ ]:
class TaskVerificationDataset(Dataset):
    """Dataset for task verification with recipe-level labels."""

    def __init__(self, embeddings_path, labels_data, recording_ids):
        self.recording_ids = recording_ids
        self.embeddings_data = np.load(embeddings_path, allow_pickle=True)
        self.labels_data = labels_data

        self.samples = []
        for rec_id in recording_ids:
            # Check if recording exists in embeddings (key is just the recording_id)
            if rec_id not in self.embeddings_data:
                continue

            # Get the full sequence of step embeddings for this recording
            sequence = self.embeddings_data[rec_id]  # Shape: (N_steps, 768)

            if len(sequence) == 0:
                continue

            label = self.labels_data.get(rec_id, {'label': 1})['label']

            self.samples.append({
                'recording_id': rec_id,
                'sequence': sequence,
                'label': label
            })

        self.max_seq_len = max(s['sequence'].shape[0] for s in self.samples) if self.samples else 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        sequence = sample['sequence']
        seq_len = sequence.shape[0]
        feature_dim = sequence.shape[1]

        # Pad sequence
        padded_sequence = np.zeros((self.max_seq_len, feature_dim), dtype=np.float32)
        padded_sequence[:seq_len] = sequence

        # Attention mask
        mask = np.zeros(self.max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0

        return {
            'sequence': torch.tensor(padded_sequence, dtype=torch.float32),
            'mask': torch.tensor(mask, dtype=torch.float32),
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }


class TransformerTaskVerifier(nn.Module):
    """Transformer-based model for task verification."""

    def __init__(self, input_dim, hidden_dim=256, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()

        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, 500, hidden_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )

    def forward(self, sequence, mask=None):
        batch_size, seq_len, _ = sequence.shape

        x = self.input_projection(sequence)
        x = x + self.pos_embedding[:, :seq_len, :]

        if mask is not None:
            attn_mask = (mask == 0)
        else:
            attn_mask = None

        x = self.transformer(x, src_key_padding_mask=attn_mask)

        # Global pooling
        if mask is not None:
            mask_expanded = mask.unsqueeze(-1)
            x_masked = x * mask_expanded
            x_pooled = x_masked.sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp(min=1)
        else:
            x_pooled = x.mean(dim=1)

        logits = self.classifier(x_pooled)
        return logits

print("✅ Task verification models defined")

✅ Task verification models defined


In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in dataloader:
        sequence = batch['sequence'].to(device)
        mask = batch['mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(sequence, mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(dataloader), accuracy_score(all_labels, all_preds)


def evaluate(model, dataloader, criterion, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in dataloader:
            sequence = batch['sequence'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(sequence, mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.0

    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }

print("✅ Training functions defined")

✅ Training functions defined


In [ ]:
# ============================================================================
# SUBSTEP 2: Train Task Verification Model (FIXED - Class Weighting + Thresholds)
# ============================================================================
# Handles class imbalance with weighted loss + tries multiple thresholds

# ===== CHOOSE STEP LOCALIZATION METHOD =====
USE_REAL_ACTIONFORMER = True  # Set to True to use real ActionFormer predictions
USE_GROUNDTRUTH = False         # Set to True to use ground-truth boundaries

# Load corresponding embeddings
if USE_REAL_ACTIONFORMER:
    embeddings_filename = 'step1_actionformer_embeddings.npz'
    method_name = "Real ActionFormer"
elif USE_GROUNDTRUTH:
    embeddings_filename = 'step1_groundtruth_embeddings.npz'
    method_name = "Ground-Truth"
else:
    embeddings_filename = 'step1_actionformer_embeddings.npz'  # old heuristic
    method_name = "Simple Heuristic"

embeddings_path = os.path.join(OUTPUT_DIR, embeddings_filename)
labels_path = os.path.join(OUTPUT_DIR, 'recipe_level_labels.json')

print(f"📂 Loading embeddings and labels...")
print(f"   Method: {method_name}")
print(f"   Embeddings: {embeddings_path}")

# Get recording IDs from the embeddings file
embeddings_data_temp = np.load(embeddings_path, allow_pickle=True)
test_recording_ids = sorted(list(set([k.split('_')[0] + '_' + k.split('_')[1] for k in embeddings_data_temp.keys() if '_' in k])))

if MAX_RECORDINGS and len(test_recording_ids) > MAX_RECORDINGS:
    test_recording_ids = test_recording_ids[:MAX_RECORDINGS]

print(f"   Found {len(test_recording_ids)} recordings in embeddings")

# ===== COMPUTE CLASS WEIGHTS =====
# Count class distribution to handle imbalance
all_labels_for_weight = [recipe_labels.get(r, {'label': 1})['label'] for r in test_recording_ids]

📂 Loading embeddings and labels...
   Method: Real ActionFormer
   Embeddings: extension_results/step1_actionformer_embeddings.npz
   Found 383 recordings in embeddings


In [ ]:
# ============================================================================
# SUBSTEP 2A: TRAIN TRANSFORMER MODEL - Leave-One-Out Cross-Validation
# ============================================================================
print("="*70)
print("🚀 TRAINING TRANSFORMER MODEL")
print("="*70)
print(f"Method: {method_name}")
print(f"Embeddings: {embeddings_filename}")

# Load embeddings
embeddings_data = np.load(embeddings_path, allow_pickle=True)

# Prepare data
all_results = []
all_preds = []
all_labels = []
all_probs = []

print(f"\n🔄 Starting Leave-One-Out Cross-Validation on {len(test_recording_ids)} recordings...")
for test_idx, test_recording in enumerate(test_recording_ids):
    print(f"\n{'='*70}")
    print(f"Fold {test_idx + 1}/{len(test_recording_ids)}: Testing on {test_recording}")
    print(f"{'='*70}")

    # Split: leave one out
    train_recordings = [r for r in test_recording_ids if r != test_recording]

    # Create datasets
    train_dataset = TaskVerificationDataset(embeddings_path, recipe_labels, train_recordings)
    test_dataset = TaskVerificationDataset(embeddings_path, recipe_labels, [test_recording])

    if len(train_dataset) == 0 or len(test_dataset) == 0:
        print(f"⚠️  Skipping {test_recording}: insufficient data")
        continue

    # Count classes in training set
    train_labels = [recipe_labels.get(r, {'label': 1})['label'] for r in train_recordings]
    n_class0 = sum(1 for l in train_labels if l == 0)
    n_class1 = sum(1 for l in train_labels if l == 1)

    print(f"Training set: {len(train_dataset)} samples (Class 0: {n_class0}, Class 1: {n_class1})")

    # Compute class weights for this fold
    if n_class0 > 0 and n_class1 > 0:
        weight_class0 = len(train_labels) / (2 * n_class0)
        weight_class1 = len(train_labels) / (2 * n_class1)
        class_weights = torch.tensor([weight_class0, weight_class1], dtype=torch.float32).to(device)
    else:
        class_weights = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)

    print(f"Class weights: {class_weights.cpu().numpy()}")

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    # Initialize model
    feature_dim = train_dataset.samples[0]['sequence'].shape[1]
    model = TransformerTaskVerifier(
        input_dim=feature_dim,
        hidden_dim=256,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ).to(device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # Training loop
    best_train_f1 = 0.0
    patience = 5
    patience_counter = 0

    for epoch in range(NUM_EPOCHS_SUBSTEP2):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS_SUBSTEP2}: Loss={train_loss:.4f}, Acc={train_acc:.3f}")

    # Evaluate on test sample
    test_metrics = evaluate(model, test_loader, criterion, device)

    print(f"\n📊 Test Results for {test_recording}:")
    print(f"   Label: {recipe_labels.get(test_recording, {'label': 1})['label']}")
    print(f"   Prediction: {all_preds[-1] if all_preds and len(all_preds) > test_idx else 'N/A'}")
    print(f"   Accuracy: {test_metrics['accuracy']:.3f}")
    print(f"   Precision: {test_metrics['precision']:.3f}")
    print(f"   Recall: {test_metrics['recall']:.3f}")
    print(f"   F1: {test_metrics['f1']:.3f}")

    # Store results
    all_results.append({
        'recording': test_recording,
        'metrics': test_metrics
    })

    # Get predictions
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            sequence = batch['sequence'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(sequence, mask)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

# ===== FINAL RESULTS =====
print(f"\n{'='*70}")
print("🎯 FINAL RESULTS - LEAVE-ONE-OUT CROSS-VALIDATION")
print(f"{'='*70}")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
final_acc = accuracy_score(all_labels, all_preds)
final_prec, final_rec, final_f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='binary', zero_division=0
)

try:
    final_auc = roc_auc_score(all_labels, all_probs)
except:
    final_auc = 0.0

print(f"\n📊 Overall Performance:")
print(f"   Accuracy:  {final_acc:.3f}")
print(f"   Precision: {final_prec:.3f}")
print(f"   Recall:    {final_rec:.3f}")
print(f"   F1 Score:  {final_f1:.3f}")
print(f"   AUC:       {final_auc:.3f}")

# Confusion matrix
tp = np.sum((all_labels == 1) & (all_preds == 1))
tn = np.sum((all_labels == 0) & (all_preds == 0))
fp = np.sum((all_labels == 0) & (all_preds == 1))
fn = np.sum((all_labels == 1) & (all_preds == 0))

print(f"\n📋 Confusion Matrix:")
print(f"   True Positives:  {tp}")
print(f"   True Negatives:  {tn}")
print(f"   False Positives: {fp}")
print(f"   False Negatives: {fn}")

print(f"\n{'='*70}")
print(f"✅ Training complete for {method_name}")
print(f"{'='*70}")

🚀 TRAINING TRANSFORMER MODEL
Method: Real ActionFormer
Embeddings: step1_actionformer_embeddings.npz

🔄 Starting Leave-One-Out Cross-Validation on 383 recordings...

Fold 1/383: Testing on 10_16
Training set: 382 samples (Class 0: 219, Class 1: 163)
Class weights: [0.8721461 1.1717792]
Epoch 1/20: Loss=0.7180, Acc=0.484
Epoch 2/20: Loss=0.6993, Acc=0.524
Epoch 4/20: Loss=0.6914, Acc=0.560
Epoch 6/20: Loss=0.6908, Acc=0.560
Epoch 8/20: Loss=0.6752, Acc=0.613
Epoch 10/20: Loss=0.6198, Acc=0.657
Epoch 12/20: Loss=0.5941, Acc=0.707
Epoch 14/20: Loss=0.5108, Acc=0.772
Epoch 16/20: Loss=0.3511, Acc=0.856
Epoch 18/20: Loss=0.1978, Acc=0.932
Epoch 20/20: Loss=0.1222, Acc=0.961

📊 Test Results for 10_16:
   Label: 1
   Prediction: N/A
   Accuracy: 1.000
   Precision: 1.000
   Recall: 1.000
   F1: 1.000

Fold 2/383: Testing on 10_18


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Training set: 382 samples (Class 0: 219, Class 1: 163)
Class weights: [0.8721461 1.1717792]
Epoch 1/20: Loss=0.7005, Acc=0.524
Epoch 2/20: Loss=0.6962, Acc=0.526
Epoch 4/20: Loss=0.6954, Acc=0.531
Epoch 6/20: Loss=0.6790, Acc=0.620
Epoch 8/20: Loss=0.6556, Acc=0.620
Epoch 10/20: Loss=0.6118, Acc=0.688
Epoch 12/20: Loss=0.5763, Acc=0.725
Epoch 14/20: Loss=0.4872, Acc=0.796
Epoch 16/20: Loss=0.3476, Acc=0.869
Epoch 18/20: Loss=0.2031, Acc=0.935
Epoch 20/20: Loss=0.1806, Acc=0.935

📊 Test Results for 10_18:
   Label: 1
   Prediction: N/A
   Accuracy: 0.000
   Precision: 0.000
   Recall: 0.000
   F1: 0.000

Fold 3/383: Testing on 10_24


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Training set: 382 samples (Class 0: 219, Class 1: 163)
Class weights: [0.8721461 1.1717792]
Epoch 1/20: Loss=0.7151, Acc=0.497
Epoch 2/20: Loss=0.7024, Acc=0.510
Epoch 4/20: Loss=0.7016, Acc=0.550
Epoch 6/20: Loss=0.6924, Acc=0.529
Epoch 8/20: Loss=0.6546, Acc=0.631
Epoch 10/20: Loss=0.6414, Acc=0.670
Epoch 12/20: Loss=0.5188, Acc=0.770
Epoch 14/20: Loss=0.4289, Acc=0.838
Epoch 16/20: Loss=0.2904, Acc=0.887
Epoch 18/20: Loss=0.1782, Acc=0.935
Epoch 20/20: Loss=0.0864, Acc=0.966

📊 Test Results for 10_24:
   Label: 1
   Prediction: N/A
   Accuracy: 0.000
   Precision: 0.000
   Recall: 0.000
   F1: 0.000

Fold 4/383: Testing on 10_26


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Training set: 382 samples (Class 0: 218, Class 1: 164)
Class weights: [0.8761468 1.1646341]
Epoch 1/20: Loss=0.7137, Acc=0.471
Epoch 2/20: Loss=0.7031, Acc=0.537
Epoch 4/20: Loss=0.6937, Acc=0.579
Epoch 6/20: Loss=0.6979, Acc=0.560
Epoch 8/20: Loss=0.6651, Acc=0.620
Epoch 10/20: Loss=0.6372, Acc=0.683
Epoch 12/20: Loss=0.5403, Acc=0.738
Epoch 14/20: Loss=0.3874, Acc=0.846
Epoch 16/20: Loss=0.2452, Acc=0.916
Epoch 18/20: Loss=0.1484, Acc=0.953
Epoch 20/20: Loss=0.0742, Acc=0.987

📊 Test Results for 9_47:
   Label: 0
   Prediction: N/A
   Accuracy: 1.000
   Precision: 0.000
   Recall: 0.000
   F1: 0.000

Fold 383/383: Testing on 9_8


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Training set: 382 samples (Class 0: 218, Class 1: 164)
Class weights: [0.8761468 1.1646341]
Epoch 1/20: Loss=0.7020, Acc=0.524
Epoch 2/20: Loss=0.7044, Acc=0.513
Epoch 4/20: Loss=0.6921, Acc=0.563
Epoch 6/20: Loss=0.6891, Acc=0.573
Epoch 8/20: Loss=0.6663, Acc=0.641
Epoch 10/20: Loss=0.6308, Acc=0.660
Epoch 12/20: Loss=0.5482, Acc=0.728
Epoch 14/20: Loss=0.3999, Acc=0.840
Epoch 16/20: Loss=0.2308, Acc=0.911
Epoch 18/20: Loss=0.0924, Acc=0.961
Epoch 20/20: Loss=0.1055, Acc=0.961

📊 Test Results for 9_8:
   Label: 0
   Prediction: N/A
   Accuracy: 0.000
   Precision: 0.000
   Recall: 0.000
   F1: 0.000

🎯 FINAL RESULTS - LEAVE-ONE-OUT CROSS-VALIDATION

📊 Overall Performance:
   Accuracy:  0.559
   Precision: 0.484
   Recall:    0.457
   F1 Score:  0.470
   AUC:       0.571

📋 Confusion Matrix:
   True Positives:  75
   True Negatives:  139
   False Positives: 80
   False Negatives: 89

✅ Training complete for Real ActionFormer


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### FOR THE GROUND TRUTH PART

In [ ]:
# ============================================================================
# SUBSTEP 2: Train Task Verification Model (FIXED - Class Weighting + Thresholds)
# ============================================================================
# Handles class imbalance with weighted loss + tries multiple thresholds

# ===== CHOOSE STEP LOCALIZATION METHOD =====
USE_REAL_ACTIONFORMER = False  # Set to True to use real ActionFormer predictions
USE_GROUNDTRUTH = True         # Set to True to use ground-truth boundaries

# Load corresponding embeddings
if USE_REAL_ACTIONFORMER:
    embeddings_filename = 'step1_actionformer_embeddings.npz'
    method_name = "Real ActionFormer"
elif USE_GROUNDTRUTH:
    embeddings_filename = 'step1_groundtruth_embeddings.npz'
    method_name = "Ground-Truth"
else:
    embeddings_filename = 'step1_actionformer_embeddings.npz'  # old heuristic
    method_name = "Simple Heuristic"

embeddings_path = os.path.join(OUTPUT_DIR, embeddings_filename)
labels_path = os.path.join(OUTPUT_DIR, 'recipe_level_labels.json')

print(f"📂 Loading embeddings and labels...")
print(f"   Method: {method_name}")
print(f"   Embeddings: {embeddings_path}")

# Get recording IDs from the embeddings file
embeddings_data_temp = np.load(embeddings_path, allow_pickle=True)
test_recording_ids = sorted(list(set([k.split('_')[0] + '_' + k.split('_')[1] for k in embeddings_data_temp.keys() if '_' in k])))

if MAX_RECORDINGS and len(test_recording_ids) > MAX_RECORDINGS:
    test_recording_ids = test_recording_ids[:MAX_RECORDINGS]

print(f"   Found {len(test_recording_ids)} recordings in embeddings")

# ===== COMPUTE CLASS WEIGHTS =====
# Count class distribution to handle imbalance
all_labels_for_weight = [recipe_labels.get(r, {'label': 1})['label'] for r in test_recording_ids]

📂 Loading embeddings and labels...
   Method: Ground-Truth
   Embeddings: extension_results/step1_groundtruth_embeddings.npz
   Found 383 recordings in embeddings


In [ ]:
# ============================================================================
# SUBSTEP 2A: TRAIN TRANSFORMER MODEL - Leave-One-Out Cross-Validation
# ============================================================================
print("="*70)
print("🚀 TRAINING TRANSFORMER MODEL")
print("="*70)
print(f"Method: {method_name}")
print(f"Embeddings: {embeddings_filename}")

# Load embeddings
embeddings_data = np.load(embeddings_path, allow_pickle=True)

# Prepare data
all_results = []
all_preds = []
all_labels = []
all_probs = []

print(f"\n🔄 Starting Leave-One-Out Cross-Validation on {len(test_recording_ids)} recordings...")
for test_idx, test_recording in enumerate(test_recording_ids):
    print(f"\n{'='*70}")
    print(f"Fold {test_idx + 1}/{len(test_recording_ids)}: Testing on {test_recording}")
    print(f"{'='*70}")

    # Split: leave one out
    train_recordings = [r for r in test_recording_ids if r != test_recording]

    # Create datasets
    train_dataset = TaskVerificationDataset(embeddings_path, recipe_labels, train_recordings)
    test_dataset = TaskVerificationDataset(embeddings_path, recipe_labels, [test_recording])

    if len(train_dataset) == 0 or len(test_dataset) == 0:
        print(f"⚠️  Skipping {test_recording}: insufficient data")
        continue

    # Count classes in training set
    train_labels = [recipe_labels.get(r, {'label': 1})['label'] for r in train_recordings]
    n_class0 = sum(1 for l in train_labels if l == 0)
    n_class1 = sum(1 for l in train_labels if l == 1)

    print(f"Training set: {len(train_dataset)} samples (Class 0: {n_class0}, Class 1: {n_class1})")

    # Compute class weights for this fold
    if n_class0 > 0 and n_class1 > 0:
        weight_class0 = len(train_labels) / (2 * n_class0)
        weight_class1 = len(train_labels) / (2 * n_class1)
        class_weights = torch.tensor([weight_class0, weight_class1], dtype=torch.float32).to(device)
    else:
        class_weights = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)

    print(f"Class weights: {class_weights.cpu().numpy()}")

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    # Initialize model
    feature_dim = train_dataset.samples[0]['sequence'].shape[1]
    model = TransformerTaskVerifier(
        input_dim=feature_dim,
        hidden_dim=256,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ).to(device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # Training loop
    best_train_f1 = 0.0
    patience = 5
    patience_counter = 0

    for epoch in range(NUM_EPOCHS_SUBSTEP2):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS_SUBSTEP2}: Loss={train_loss:.4f}, Acc={train_acc:.3f}")

    # Evaluate on test sample
    test_metrics = evaluate(model, test_loader, criterion, device)

    print(f"\n📊 Test Results for {test_recording}:")
    print(f"   Label: {recipe_labels.get(test_recording, {'label': 1})['label']}")
    print(f"   Prediction: {all_preds[-1] if all_preds and len(all_preds) > test_idx else 'N/A'}")
    print(f"   Accuracy: {test_metrics['accuracy']:.3f}")
    print(f"   Precision: {test_metrics['precision']:.3f}")
    print(f"   Recall: {test_metrics['recall']:.3f}")
    print(f"   F1: {test_metrics['f1']:.3f}")

    # Store results
    all_results.append({
        'recording': test_recording,
        'metrics': test_metrics
    })

    # Get predictions
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            sequence = batch['sequence'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(sequence, mask)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

# ===== FINAL RESULTS =====
print(f"\n{'='*70}")
print("🎯 FINAL RESULTS - LEAVE-ONE-OUT CROSS-VALIDATION")
print(f"{'='*70}")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
final_acc = accuracy_score(all_labels, all_preds)
final_prec, final_rec, final_f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='binary', zero_division=0
)

try:
    final_auc = roc_auc_score(all_labels, all_probs)
except:
    final_auc = 0.0

print(f"\n📊 Overall Performance:")
print(f"   Accuracy:  {final_acc:.3f}")
print(f"   Precision: {final_prec:.3f}")
print(f"   Recall:    {final_rec:.3f}")
print(f"   F1 Score:  {final_f1:.3f}")
print(f"   AUC:       {final_auc:.3f}")

# Confusion matrix
tp = np.sum((all_labels == 1) & (all_preds == 1))
tn = np.sum((all_labels == 0) & (all_preds == 0))
fp = np.sum((all_labels == 0) & (all_preds == 1))
fn = np.sum((all_labels == 1) & (all_preds == 0))

print(f"\n📋 Confusion Matrix:")
print(f"   True Positives:  {tp}")
print(f"   True Negatives:  {tn}")
print(f"   False Positives: {fp}")
print(f"   False Negatives: {fn}")

print(f"\n{'='*70}")
print(f"✅ Training complete for {method_name}")
print(f"{'='*70}")

🚀 TRAINING TRANSFORMER MODEL
Method: Ground-Truth
Embeddings: step1_groundtruth_embeddings.npz

🔄 Starting Leave-One-Out Cross-Validation on 383 recordings...

Fold 1/383: Testing on 10_16
Training set: 382 samples (Class 0: 219, Class 1: 163)
Class weights: [0.8721461 1.1717792]
Epoch 1/20: Loss=0.7217, Acc=0.490
Epoch 2/20: Loss=0.6962, Acc=0.563
Epoch 4/20: Loss=0.6821, Acc=0.545
Epoch 6/20: Loss=0.6752, Acc=0.599
Epoch 8/20: Loss=0.6729, Acc=0.563
Epoch 10/20: Loss=0.6212, Acc=0.654
Epoch 12/20: Loss=0.5749, Acc=0.723
Epoch 14/20: Loss=0.5095, Acc=0.775
Epoch 16/20: Loss=0.4151, Acc=0.830
Epoch 18/20: Loss=0.4279, Acc=0.832
Epoch 20/20: Loss=0.3837, Acc=0.851

📊 Test Results for 10_16:
   Label: 1
   Prediction: N/A
   Accuracy: 1.000
   Precision: 1.000
   Recall: 1.000
   F1: 1.000

Fold 2/383: Testing on 10_18
Training set: 382 samples (Class 0: 219, Class 1: 163)
Class weights: [0.8721461 1.1717792]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 1/20: Loss=0.7114, Acc=0.503
Epoch 2/20: Loss=0.6973, Acc=0.558
Epoch 4/20: Loss=0.6782, Acc=0.592
Epoch 6/20: Loss=0.6473, Acc=0.620
Epoch 8/20: Loss=0.5985, Acc=0.675
Epoch 10/20: Loss=0.5320, Acc=0.751
Epoch 12/20: Loss=0.4599, Acc=0.788
Epoch 14/20: Loss=0.3590, Acc=0.840
Epoch 16/20: Loss=0.2637, Acc=0.916
Epoch 18/20: Loss=0.2299, Acc=0.916
Epoch 20/20: Loss=0.1962, Acc=0.935

📊 Test Results for 10_18:
   Label: 1
   Prediction: N/A
   Accuracy: 1.000
   Precision: 1.000
   Recall: 1.000
   F1: 1.000

Fold 3/383: Testing on 10_24
Training set: 382 samples (Class 0: 219, Class 1: 163)
Class weights: [0.8721461 1.1717792]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 1/20: Loss=0.7110, Acc=0.503
Epoch 2/20: Loss=0.6979, Acc=0.524
Epoch 4/20: Loss=0.6830, Acc=0.550
Epoch 6/20: Loss=0.6551, Acc=0.615
Epoch 8/20: Loss=0.6569, Acc=0.623
Epoch 10/20: Loss=0.5928, Acc=0.691
Epoch 12/20: Loss=0.4932, Acc=0.754
Epoch 14/20: Loss=0.4861, Acc=0.785
Epoch 16/20: Loss=0.3417, Acc=0.851
Epoch 18/20: Loss=0.2873, Acc=0.880
Epoch 20/20: Loss=0.2395, Acc=0.914

📊 Test Results for 10_24:
   Label: 1
   Prediction: N/A
   Accuracy: 1.000
   Precision: 1.000
   Recall: 1.000
   F1: 1.000

Fold 4/383: Testing on 10_26
Training set: 382 samples (Class 0: 218, Class 1: 164)
Class weights: [0.8761468 1.1646341]


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 1/20: Loss=0.7103, Acc=0.531
Epoch 2/20: Loss=0.7265, Acc=0.463
Epoch 4/20: Loss=0.6934, Acc=0.565
Epoch 6/20: Loss=0.6860, Acc=0.592
Epoch 8/20: Loss=0.6373, Acc=0.657
Epoch 10/20: Loss=0.5928, Acc=0.681
Epoch 12/20: Loss=0.5484, Acc=0.712
Epoch 14/20: Loss=0.5258, Acc=0.730
Epoch 16/20: Loss=0.4100, Acc=0.830
Epoch 18/20: Loss=0.4105, Acc=0.819
Epoch 20/20: Loss=0.2949, Acc=0.861

📊 Test Results for 9_47:
   Label: 0
   Prediction: N/A
   Accuracy: 1.000
   Precision: 0.000
   Recall: 0.000
   F1: 0.000

Fold 383/383: Testing on 9_8
Training set: 382 samples (Class 0: 218, Class 1: 164)
Class weights: [0.8761468 1.1646341]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 1/20: Loss=0.7145, Acc=0.492
Epoch 2/20: Loss=0.6999, Acc=0.516
Epoch 4/20: Loss=0.6816, Acc=0.565
Epoch 6/20: Loss=0.6775, Acc=0.576
Epoch 8/20: Loss=0.6384, Acc=0.654
Epoch 10/20: Loss=0.6191, Acc=0.673
Epoch 12/20: Loss=0.5520, Acc=0.725
Epoch 14/20: Loss=0.4556, Acc=0.788
Epoch 16/20: Loss=0.4410, Acc=0.806
Epoch 18/20: Loss=0.3255, Acc=0.887
Epoch 20/20: Loss=0.2674, Acc=0.916

📊 Test Results for 9_8:
   Label: 0
   Prediction: N/A
   Accuracy: 0.000
   Precision: 0.000
   Recall: 0.000
   F1: 0.000

🎯 FINAL RESULTS - LEAVE-ONE-OUT CROSS-VALIDATION

📊 Overall Performance:
   Accuracy:  0.645
   Precision: 0.583
   Recall:    0.598
   F1 Score:  0.590
   AUC:       0.674

📋 Confusion Matrix:
   True Positives:  98
   True Negatives:  149
   False Positives: 70
   False Negatives: 66

✅ Training complete for Ground-Truth


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
